# D+ → π− π+ π+ Cartesian toy-MC fit closure

Reference closure for the coherent model:

`A = c_rho F_rho + c_f0 F_f0 + c_NR`, with `c_i = x_i + i y_i`.

This notebook uses **100,000 fit events** and **1,000,000 normalization events**. The reference coefficient is fixed to `c_rho = 1 + 0i`. The `f0` and NR coefficients are fitted directly in Cartesian coordinates, with no CP parameters.

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import mplhep as hep

from dalitzplotfitter import ThreeBodyPhaseSpace, enable_x64
from dalitzplotfitter.amplitude import (
    AmplitudeBuilder, AmplitudeComponent, PreparedAmplitudeCache,
    ConstantAmplitude, compile_amplitude_component,
    create_kinematic_transformer,
)
from dalitzplotfitter.coefficients import FitCartesian
from dalitzplotfitter.fit import Minimizer, Parameter
from dalitzplotfitter.reaction import ReactionBuilder
from dalitzplotfitter.toy import ToyGenerator

enable_x64()
hep.style.use("LHCb2")

FIT_SAMPLE_SIZE = 100_000
NORMALIZATION_SAMPLE_SIZE = 1_000_000

In [ ]:
def build_resonance(resonance):
    reaction = ReactionBuilder(
        initial_state="D+",
        final_state=["pi-", "pi+", "pi+"],
        allowed_intermediate_particles=[resonance],
    ).build()
    model = AmplitudeBuilder(reaction).build()
    return reaction, model, compile_amplitude_component(model)

rho_reaction, rho_model, rho_dynamics = build_resonance("rho(770)0")
_, _, f0_dynamics = build_resonance("f(0)(980)")

In [ ]:
truth = {
    "f0.x": 0.55 * np.cos(1.15),
    "f0.y": 0.55 * np.sin(1.15),
    "nr.x": 0.28 * np.cos(-0.85),
    "nr.y": 0.28 * np.sin(-0.85),
}

rho_x = Parameter.coefficient("rho.x", 1.0, fixed=True, owner="rho")
rho_y = Parameter.coefficient("rho.y", 0.0, fixed=True, owner="rho")

rng = np.random.default_rng(314159)
f0_x = Parameter.coefficient("f0.x", float(rng.uniform(-0.8, 0.8)), bounds=(-1.5, 1.5), step=0.02, owner="f0")
f0_y = Parameter.coefficient("f0.y", float(rng.uniform(-0.8, 0.8)), bounds=(-1.5, 1.5), step=0.02, owner="f0")
nr_x = Parameter.coefficient("nr.x", float(rng.uniform(-0.6, 0.6)), bounds=(-1.0, 1.0), step=0.02, owner="NR")
nr_y = Parameter.coefficient("nr.y", float(rng.uniform(-0.6, 0.6)), bounds=(-1.0, 1.0), step=0.02, owner="NR")

parameters = (rho_x, rho_y, f0_x, f0_y, nr_x, nr_y)
initial = {p.name: p.value for p in parameters}

rho_coefficient = FitCartesian(rho_x, rho_y)
f0_coefficient = FitCartesian(f0_x, f0_y)
nr_coefficient = FitCartesian(nr_x, nr_y)

print("Truth:", truth)
print("Initial:", {k: initial[k] for k in truth})

In [ ]:
components = (
    AmplitudeComponent("rho", rho_dynamics, rho_coefficient),
    AmplitudeComponent("f0", f0_dynamics, f0_coefficient),
    AmplitudeComponent("NR", ConstantAmplitude(), nr_coefficient),
)

phase_space = ThreeBodyPhaseSpace.from_reaction(rho_reaction)
transformer = create_kinematic_transformer(rho_model)

def toy_intensity(data, values):
    amplitude = (
        rho_coefficient.value(values=values) * rho_dynamics(data, None)
        + f0_coefficient.value(values=values) * f0_dynamics(data, None)
        + nr_coefficient.value(values=values) * ConstantAmplitude()(data, None)
    )
    return jnp.real(amplitude * jnp.conj(amplitude))

generator = ToyGenerator(
    phase_space=phase_space,
    transformer=transformer,
    envelope_safety=1.2,
)
toy_sample, toy_data = generator.generate(
    jax.random.key(2026),
    size=FIT_SAMPLE_SIZE,
    intensity=toy_intensity,
    parameters=truth,
)
print(f"Generated {toy_sample.size:,} fit events")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
h = ax.hist2d(np.asarray(toy_sample.s12), np.asarray(toy_sample.s13), bins=100)
fig.colorbar(h[3], ax=ax, label="Candidates")
ax.set_xlabel(r"$m^2(\pi^-\pi^+_1)$ [GeV$^2$]")
ax.set_ylabel(r"$m^2(\pi^-\pi^+_2)$ [GeV$^2$]")
ax.set_title(r"Toy $D^+\to\pi^-\pi^+\pi^+$")
plt.show()

In [ ]:
normalization_sample = phase_space.generate(
    jax.random.key(2027),
    NORMALIZATION_SAMPLE_SIZE,
)
normalization_data = transformer(normalization_sample.as_momentum_dict())
print(f"Generated {normalization_sample.size:,} normalization events")

cache = PreparedAmplitudeCache.prepare(
    components,
    data=toy_data,
    normalization_data=normalization_data,
    normalization_weights=normalization_sample.weights,
    parameters=parameters,
)

def nll(values):
    intensity, normalization = cache.evaluate(values)
    return -jnp.sum(jnp.log(jnp.clip(intensity, min=1e-300))) + toy_sample.size * jnp.log(normalization)

print("NLL truth  :", float(nll(truth)))
print("NLL initial:", float(nll(initial)))

In [ ]:
minimizer = Minimizer(nll, parameters)
starts = (
    initial,
    truth,
    {"f0.x": 0.2, "f0.y": 0.5, "nr.x": 0.2, "nr.y": -0.2},
    {"f0.x": -0.5, "f0.y": 0.5, "nr.x": 0.3, "nr.y": 0.3},
    {"f0.x": 0.5, "f0.y": -0.5, "nr.x": -0.3, "nr.y": -0.3},
)
results = [minimizer.fit(start_values=start) for start in starts]
valid_results = [result for result in results if result.valid]
result = min(valid_results, key=lambda candidate: float(candidate.fval))

fit_values = {name: float(result.values[name]) for name in truth}
fit_errors = {name: float(result.errors[name]) for name in truth}

print(result)
for name in truth:
    print(
        f"{name:6s}: truth={truth[name]:+.5f}  "
        f"fit={fit_values[name]:+.5f} ± {fit_errors[name]:.5f}"
    )
print("NLL truth:", float(nll(truth)))
print("NLL best :", float(result.fval))
print("Delta NLL truth-best:", float(nll(truth)) - float(result.fval))

In [ ]:
def model_weights(values):
    c = cache.coefficient_vector(values)
    amp = cache.normalization_components @ c
    intensity = jnp.real(amp * jnp.conj(amp))
    return np.asarray(normalization_sample.weights * intensity)

weights_initial = model_weights(initial)
weights_fit = model_weights(fit_values)

vars_ = [
    (toy_sample.s12, normalization_sample.s12, r"$m^2(\pi^-\pi^+_1)$ [GeV$^2$]"),
    (toy_sample.s13, normalization_sample.s13, r"$m^2(\pi^-\pi^+_2)$ [GeV$^2$]"),
    (toy_sample.s23, normalization_sample.s23, r"$m^2(\pi^+_1\pi^+_2)$ [GeV$^2$]"),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (toy_x, mc_x, xlabel) in zip(axes, vars_):
    counts, edges = np.histogram(np.asarray(toy_x), bins=60)
    centers = 0.5 * (edges[:-1] + edges[1:])
    h0, _ = np.histogram(np.asarray(mc_x), bins=edges, weights=weights_initial)
    h1, _ = np.histogram(np.asarray(mc_x), bins=edges, weights=weights_fit)
    h0 *= counts.sum() / h0.sum()
    h1 *= counts.sum() / h1.sum()
    ax.errorbar(centers, counts, yerr=np.sqrt(counts), fmt="o", label="Toy")
    ax.stairs(h0, edges, label="Before fit")
    ax.stairs(h1, edges, label="After fit")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Candidates")
    ax.legend()
fig.suptitle("Toy projections: before and after the Cartesian fit")
fig.tight_layout()
plt.show()

In [ ]:
bins = 80
xedges = np.linspace(
    min(float(jnp.min(toy_sample.s12)), float(jnp.min(normalization_sample.s12))),
    max(float(jnp.max(toy_sample.s12)), float(jnp.max(normalization_sample.s12))),
    bins + 1,
)
yedges = np.linspace(
    min(float(jnp.min(toy_sample.s13)), float(jnp.min(normalization_sample.s13))),
    max(float(jnp.max(toy_sample.s13)), float(jnp.max(normalization_sample.s13))),
    bins + 1,
)
toy_h, _, _ = np.histogram2d(np.asarray(toy_sample.s12), np.asarray(toy_sample.s13), bins=[xedges, yedges])
start_h, _, _ = np.histogram2d(
    np.asarray(normalization_sample.s12), np.asarray(normalization_sample.s13),
    bins=[xedges, yedges], weights=weights_initial,
)
fit_h, _, _ = np.histogram2d(
    np.asarray(normalization_sample.s12), np.asarray(normalization_sample.s13),
    bins=[xedges, yedges], weights=weights_fit,
)

toy_h /= toy_h.sum()
start_h /= start_h.sum()
fit_h /= fit_h.sum()
vmax = max(toy_h.max(), start_h.max(), fit_h.max())

fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))
for ax, hist, title in zip(
    axes,
    [toy_h, start_h, fit_h],
    ["Toy data", "Model before fit", "Model after fit"],
):
    mesh = ax.pcolormesh(xedges, yedges, hist.T, shading="auto", vmin=0, vmax=vmax)
    ax.set_xlabel(r"$m^2(\pi^-\pi^+_1)$ [GeV$^2$]")
    ax.set_ylabel(r"$m^2(\pi^-\pi^+_2)$ [GeV$^2$]")
    ax.set_title(title)
fig.colorbar(mesh, ax=axes, label="Normalized bin content")
plt.show()

The Cartesian parameterization removes the periodic phase boundary from the minimization. A successful closure should recover the injected `x` and `y` coordinates and show agreement between the 100k-event toy and the model built from the independent 1M-event normalization sample.